# LightGBM

Un seul modèle pour les 3 arcs (`axe` est une feature). À tester : hyperparamètres, `features="toutes"` dans `split`, un modèle par arc, cible en ratio…

In [10]:
import sys; sys.path.append("..")  # pour importer dataset_create depuis notebooks/
import numpy as np
import pandas as pd
from dataset_create import charger, split, ORIGINES, FEATURES

df = charger(imputer=True)  # imputer=False : trous laissés en NaN
rmse = lambda y, p, axe: ((y - p) ** 2).groupby(axe, observed=True).mean() ** .5  # RMSE par arc

In [11]:
import lightgbm as lgb

PARAMS = dict(n_estimators=500, learning_rate=0.05, num_leaves=31, min_child_samples=20)  # <- à changer à la main

def modele(X_train, y_train, X_test):
    return lgb.LGBMRegressor(verbose=-1, **PARAMS).fit(X_train, y_train).predict(X_test)

In [12]:
# 1 semaine : dernière donnée le ven. 07/11/2025 -> on prédit sam. 08 -> mer. 12/11 (11/11 férié inclus)
CIBLE = "debit"  # ou "occupation"
X_train, X_test, y_train, y_test = split(df, "2025-11-07", cible=CIBLE)
p = modele(X_train, y_train, X_test)
print(pd.DataFrame({"modele": rmse(y_test, p, X_test.axe), "naif_lag7j": rmse(y_test, X_test.lag_7j, X_test.axe)}).round(1))

                modele  naif_lag7j
axe                               
Champs-Elysees   131.0       266.5
Convention        60.3        90.3
Saints-Peres      41.4        72.3


In [13]:
# Backtest : même chose sur 1 vendredi sur 4 (ORIGINES pour tous), puis moyenne
res, err = [], []
for t0 in ORIGINES[::4]:
    X_train, X_test, y_train, y_test = split(df, t0, cible=CIBLE)
    if X_test.empty:  # aucun arc avec assez de vraies valeurs cette semaine-là
        continue
    p = modele(X_train, y_train, X_test)
    res.append(pd.DataFrame({"modele": rmse(y_test, p, X_test.axe), "naif_lag7j": rmse(y_test, X_test.lag_7j, X_test.axe)}))
    err.append(pd.DataFrame({"axe": X_test.axe, "jour": X_test.type_jour, "e2": (y_test - p) ** 2}))
par_arc = pd.concat(res).groupby(level=0).mean()
print(par_arc.round(1), "\n\nMoyenne des 3 arcs :\n", par_arc.mean().round(1))

# Erreur par type de jour (0 = semaine, 1 = samedi, 2 = dimanche ou férié)
print("\n", (pd.concat(err).groupby(["jour", "axe"], observed=True).e2.mean() ** .5).unstack().round(0))

                modele  naif_lag7j
axe                               
Champs-Elysees   131.4       168.2
Convention        71.3        83.4
Saints-Peres      69.3        80.9 

Moyenne des 3 arcs :
 modele         90.7
naif_lag7j    110.8
dtype: float64

 axe   Champs-Elysees  Convention  Saints-Peres
jour                                          
0              143.0        78.0          74.0
1              103.0        62.0          65.0
2              165.0        67.0          78.0


In [ ]:
# Entraîner seulement à partir d'une année : erreur sur toutes les ORIGINES, par arc et par type de jour
# (les features restent calculées sur tout l'historique : lag_an, profils… ; seules les heures d'entraînement changent)
ANNEES = [2014, 2016, 2018, 2020, 2022, 2023, 2024]  # début du train : 1er janvier de cette année
err = []
for t0 in ORIGINES:
    X_train, X_test, y_train, y_test = split(df, t0, cible=CIBLE)
    if X_test.empty:
        continue
    jour = np.select([X_test.ferie == 1, X_test.jour_sem >= 5], ["ferie", "w-e"], "semaine")
    an_train = df.datetime.loc[X_train.index].dt.year  # année de chaque heure du train
    for an in ANNEES:
        p = modele(X_train[an_train >= an], y_train[an_train >= an], X_test)
        err.append(pd.DataFrame({"annee": an, "arc": X_test.axe.astype(str), "jour": jour, "e2": (y_test - p) ** 2}))
E = pd.concat(err)
rmse_par = lambda *g: (E.groupby(["annee", *g]).e2.mean() ** .5).unstack(list(g)).round(1)  # RMSE sur toutes les heures de val
print(rmse_par("arc").assign(moyenne=lambda t: t.mean(axis=1).round(1)), "\n")
print(rmse_par("arc", "jour").to_string(), "\n")
print("heures de val par type de jour :", E[E.annee == ANNEES[0]].jour.value_counts().to_dict())